# Cabezas encadenadas en el modelo final

**Inferencia de propiedades físicas de galaxias a partir de imágenes multibanda** · Instituto Tecnológico y de Estudios Superiores de Monterrey · Maestría en Inteligencia Artificial Aplicada · **Equipo 24** · Colaboración con el Instituto de Astronomía de la UNAM

| Integrante | Matrícula |
|---|---|
| Luz Copelia Minutti Pérez | A01796921 |
| Erick Alberto Bustos Cruz | A01378966 |
| José Florencio Maguey Peralta | A01796727 |

**Asesores:** Dr. Iván Reyes Amezcua (ITESM) · Dr. Héctor Hernández Toledo y Dr. José Antonio Vázquez Mata (IA-UNAM)

**Fase de CRISP-ML(Q):** 3, *Machine Learning Model Engineering* (Studer et al. 2021). Experimento posterior al modelo final del Avance 5.

## 1. Planteamiento

El modelo final del Avance 5 combina tres arquitecturas, DINOv3 con LoRA, ConvNeXt-Tiny y EfficientNet-B3, cada una entrenada con tres semillas y con el redshift espectroscópico como entrada. En las tres, la imagen se convierte en un vector, se le une el redshift y tres cabezas de regresión leen ese mismo vector en paralelo: una estima la masa estelar, otra la SFR y otra el D4000. Las cabezas comparten el extractor, pero ninguna ve lo que predicen las otras.

Katachi (Alfonzo et al. 2024), el antecedente de este proyecto, organiza las tres propiedades en cadena: una red estima la masa, una segunda red recibe esa masa para estimar la SFR y una tercera recibe ambas para estimar el D4000. La motivación es física. La SFR de las galaxias que forman estrellas sigue de cerca a su masa (la secuencia principal) y el D4000 crece con la masa y con la edad de la población estelar, así que conocer la masa reduce lo que queda por explicar en las otras dos propiedades. En este cuaderno probamos esa idea sobre nuestro modelo: conservamos todo lo demás y solo encadenamos las cabezas.

No es obvio que la cadena ayude. En Katachi cada propiedad tiene su propia red, de modo que la red de la SFR no tendría acceso a la masa si no se la pasaran. En nuestro modelo las tres cabezas leen el mismo vector, que ya contiene la información con la que se estima la masa. Nuestra hipótesis es que la cadena ayuda poco en promedio y, si ayuda, lo hace en la SFR de las galaxias con formación estelar muy baja, que es donde el modelo del Avance 5 comete sus errores más grandes. El experimento está diseñado para distinguir entre las dos posibilidades.

| Identificador | Arquitectura | Entradas | Cabezas | Origen | Papel |
|---|---|---|---|---|---|
| `dinov3_z` | DINOv3 con LoRA | Imagen en color de Lupton y redshift | Independientes | Avance 5 | Referencia |
| `convnext_z` | ConvNeXt-Tiny | Bandas g, r, z y redshift | Independientes | Avance 5 | Referencia |
| `efficientnet_z` | EfficientNet-B3 | Bandas g, r, z y redshift | Independientes | Avance 5 | Referencia |
| `dinov3_z_cadena` | DINOv3 con LoRA | Igual que `dinov3_z` | En cadena | Este cuaderno | Candidato |
| `convnext_z_cadena` | ConvNeXt-Tiny | Igual que `convnext_z` | En cadena | Este cuaderno | Candidato |
| `efficientnet_z_cadena` | EfficientNet-B3 | Igual que `efficientnet_z` | En cadena | Este cuaderno | Candidato |
| Mezcla convexa, cabezas independientes | Las tres familias del Avance 5 | Predicciones de nueve redes | Independientes | Avance 5 | Modelo final vigente |
| Mezcla convexa, cabezas en cadena | Las tres familias encadenadas | Predicciones de nueve redes | En cadena | Este cuaderno | Candidato |
| Mezcla convexa de las seis familias | Independientes y encadenadas | Predicciones de dieciocho redes | Ambas | Este cuaderno | Candidato |

La cadena sigue el orden de Katachi. Cada cabeza tiene la misma forma que en el Avance 5 (una capa oculta de 256 unidades con dropout) y solo cambia lo que recibe:

| Cabeza | Recibe | Estima |
|---|---|---|
| 1 | Vector de la imagen y redshift | log M\* |
| 2 | Vector de la imagen, redshift y log M\* estimada | log SFR |
| 3 | Vector de la imagen, redshift, log M\* y log SFR estimadas | D4000 |

Dos decisiones de diseño definen el experimento. La primera es que cada cabeza recibe la **estimación** de las anteriores y nunca la etiqueta. Si en el entrenamiento la cabeza de la SFR recibiera la masa medida, aprendería a confiar en un dato exacto que no tendrá con galaxias nuevas, donde solo existe la masa estimada; es el mismo problema que resolvimos en el Avance 4 al usar un redshift estimado como entrada. La estimación de las galaxias de entrenamiento es algo más precisa que la de galaxias nuevas, porque la red ya las vio; si eso hiciera que la cadena confiara demasiado en la masa, lo veríamos en validación como un aumento del error de la SFR. La segunda es que, por defecto, el error de la SFR y del D4000 **no corrige** a las cabezas anteriores (el gradiente no atraviesa la cadena). Así cada cabeza aprende solo su propiedad, como las redes separadas de Katachi, y cualquier cambio en la masa se puede atribuir al entrenamiento y no a que la cabeza de masa se deforme para ayudar a las demás. La variable `CON_GRADIENTE` permite probar la alternativa.

El protocolo es el del Avance 5. Cada red conserva el extractor, el preentrenamiento, la imagen de entrada, los hiperparámetros y las semillas de su versión con cabezas independientes, que leemos de sus corridas guardadas; la única diferencia es la cabeza. Comparamos cada familia con su versión independiente sobre las mismas galaxias de validación mediante *bootstrap* pareado, y reconstruimos la mezcla convexa con la misma validación anidada y la misma rejilla de regularización que en el Avance 5. Solo reemplazamos el modelo final si un candidato lo supera por más de un error estándar de la diferencia pareada; registramos la elección antes de mirar la prueba.

La prueba ya se abrió una vez para el modelo final del Avance 5. Una segunda evaluación con un modelo distinto ya no es una prueba independiente en sentido estricto, porque el experimento existe después de conocer aquel resultado. Por eso solo evaluamos en prueba si la cadena gana en validación, y en ese caso lo reportamos como una confirmación del resultado de validación.

El cuaderno se organiza así: la sección 2 fija los parámetros, las secciones 3 y 4 preparan el entorno y los datos, la 5 recupera la configuración de los miembros del Avance 5, la 6 entrena las cabezas encadenadas, la 7 compara cada familia con su versión independiente, la 8 examina dónde actúa la cadena, la 9 reconstruye los ensambles, la 10 aplica la regla de selección y la 11 evalúa en prueba si corresponde.

## 2. Parámetros

| Parámetro | Valor | Significado |
|---|---|---|
| `ENCADENAR` | sí | Interruptor del experimento: `True` entrena las cabezas en cadena; `False` recupera las redes del Avance 5 sin entrenarlas, lo que sirve para comprobar que el cuaderno reproduce sus predicciones |
| `CON_GRADIENTE` | no | Si el error de la SFR y del D4000 también corrige a las cabezas anteriores de la cadena |
| `SEMILLAS` | 42, 7, 2026 | Las mismas del Avance 5 |
| `USAR_TTA` | sí | Las predicciones promedian las 8 simetrías de la imagen, como en el modelo final |
| `PLIEGUES_EXTERNOS`, `REPETICIONES`, `PLIEGUES_INTERNOS` | 5, 3, 4 | Validación anidada de los ensambles, igual que en el Avance 5 |
| `N_REMUESTREOS` | 2000 | Remuestreos *bootstrap* de los intervalos de confianza |
| `MARGEN` | 0.02, 0.02, 0.01 | Diferencia de RMSE que consideramos despreciable en log M\*, log SFR y D4000 |
| `EVALUAR_PRUEBA` | sí | Si se evalúa en prueba el modelo elegido cuando este no es el del Avance 5 |

In [ ]:
import os
variable_entorno = os.environ.get          # las variables MA_* solo se usan en la prueba local reducida

# ------------------------------------------------------------------------------------------------
# INTERRUPTOR DEL EXPERIMENTO: True entrena las cabezas en cadena; False recupera las cabezas independientes del Avance 5.
ENCADENAR = True
CON_GRADIENTE = False      # True: el error de la SFR y del D4000 también corrige a las cabezas anteriores
# ------------------------------------------------------------------------------------------------
ENCADENAR = variable_entorno("MA_ENCADENAR", str(int(ENCADENAR))) == "1"
CON_GRADIENTE = variable_entorno("MA_CON_GRADIENTE", str(int(CON_GRADIENTE))) == "1"
if CON_GRADIENTE and not ENCADENAR:
    raise ValueError("CON_GRADIENTE solo aplica cuando ENCADENAR = True")
SUFIJO = ("_cadena" + ("_gradiente" if CON_GRADIENTE else "")) if ENCADENAR else "_verificacion"
ETIQUETA_NUEVA = ("en cadena" + (" con gradiente" if CON_GRADIENTE else "")) if ENCADENAR else "recuperadas (verificación)"
FAMILIAS_BASE = ("dinov3_z", "convnext_z", "efficientnet_z")
FAMILIAS_NUEVAS = {f: f + SUFIJO for f in FAMILIAS_BASE}
VARIABLES_ESCALARES = ("redshift",)

SEMILLAS = tuple(int(s) for s in variable_entorno("MA_SEMILLAS", "42,7,2026").split(","))
USAR_TTA = variable_entorno("MA_TTA", "1") == "1"
EVALUAR_PRUEBA = variable_entorno("MA_EVALUAR_PRUEBA", "1") == "1"
N_REMUESTREOS = int(variable_entorno("MA_REMUESTREOS", "2000"))
PLIEGUES_EXTERNOS = int(variable_entorno("MA_PLIEGUES_EXTERNOS", "5"))
REPETICIONES = int(variable_entorno("MA_REPETICIONES", "3"))
PLIEGUES_INTERNOS = int(variable_entorno("MA_PLIEGUES_INTERNOS", "4"))
SEMILLA_ANIDADA = 7
N_TRABAJOS = int(variable_entorno("MA_TRABAJOS", "-1"))
LIMITE_GALAXIAS = int(variable_entorno("MA_LIMITE", "0"))           # 0 = todas las galaxias

from pathlib import Path
BASE_DRIVE = Path(variable_entorno("MA_BASE", "/content/drive/MyDrive"))
CARPETA_RESULTADOS = "resultados_modelos_alternativos"
NOMBRE_MODELO = "avance5_modelos_base"            # las corridas encadenadas se guardan junto a las del Avance 5
NOMBRE_SALIDA = "experimento_cabezas_encadenadas"
OBJETIVOS = ("log_mstar", "log_sfr", "d4000")
NOMBRES = {"log_mstar": "log M* [dex]", "log_sfr": "log SFR [dex]", "d4000": "D4000"}
CORTO = {"log_mstar": "M*", "log_sfr": "SFR", "d4000": "D4000"}
MARGEN = {"M*": 0.02, "SFR": 0.02, "D4000": 0.01}
CORTE_SSFR = -10.8                                                   # frontera entre galaxias formadoras y apagadas
CANDIDATOS_MEZCLA = [{"tipo": "mezcla_convexa", "lambda": l, "alcance": a} for a in ("por_objetivo", "global") for l in (0.0, 1e-3, 1e-2, 1e-1, 1.0, 10.0)]
print(f"Cabezas {ETIQUETA_NUEVA} · familias {list(FAMILIAS_NUEVAS.values())} · semillas {SEMILLAS}")

## 3. Entorno

Descargamos el código del repositorio del equipo, de modo que cada resultado quede ligado a una versión concreta, e instalamos `timm`, la biblioteca de modelos preentrenados. Este cuaderno necesita una GPU; recomendamos una A100. Si la sesión se interrumpe, al volver a ejecutarlo se retoman los entrenamientos ya terminados.

In [ ]:
import sys, json, time, subprocess, platform, warnings, zipfile
from dataclasses import replace
from datetime import datetime, timezone

EN_COLAB = "google.colab" in sys.modules
URL_REPOSITORIO = "https://github.com/erteck/InferenciaPropiedadesFisicas.git"
CARPETA_REPOSITORIO = Path(variable_entorno("MA_CARPETA_REPOSITORIO", "/content/InferenciaPropiedadesFisicas"))
if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm>=1.0.20", "hickle"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    if not CARPETA_REPOSITORIO.exists():
        subprocess.run(["git", "clone", "-q", URL_REPOSITORIO, str(CARPETA_REPOSITORIO)], check=True)
    subprocess.run(["git", "-C", str(CARPETA_REPOSITORIO), "pull", "-q"], check=True)
sys.path.insert(0, str(CARPETA_REPOSITORIO))
COMMIT = subprocess.run(["git", "-C", str(CARPETA_REPOSITORIO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch, timm
import matplotlib.pyplot as plt
from IPython.display import display

from src.modelos_alternativos import ensambles, evaluacion, experimento, predicciones
from src.modelos_alternativos.arquitecturas import CabezaEncadenada, EspecificacionModelo
from src.modelos_alternativos.contexto import preparar_contexto
from src.modelos_alternativos.entrenamiento import ConfiguracionEntrenamiento

DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
NOMBRE_GPU = torch.cuda.get_device_name(0) if DISPOSITIVO.type == "cuda" else "CPU"
CARPETA_PREDICCIONES_AVANCE5 = Path(variable_entorno("MA_CARPETA_PREDICCIONES_AVANCE5",
                                                     str(CARPETA_REPOSITORIO / "Avance 5. Modelo final" / "predicciones")))
RESUMEN_HETEROGENEOS_AVANCE5 = CARPETA_PREDICCIONES_AVANCE5.parent / "ensambles_heterogeneos" / "heterogeneos_con_redshift.json"
pd.set_option("display.precision", 4, "display.max_columns", 40, "display.width", 220)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})

def liberar_gpu():
    if DISPOSITIVO.type == "cuda":
        torch.cuda.empty_cache()

def registro_compacto(mensaje):
    # Durante el entrenamiento solo se muestra el avance cada 5 épocas.
    if mensaje.startswith("época") and int(mensaje.split()[1]) % 5 == 0:
        print(mensaje)

print("Python", platform.python_version(), "· commit", COMMIT[:12], "· timm", timm.__version__, "·", NOMBRE_GPU)

## 4. Datos

Preparamos la muestra y las transformaciones con las reglas del protocolo común: las escalas de las imágenes y las medias y desviaciones de las etiquetas y del redshift se calculan solo con las galaxias de entrenamiento. Como el experimento compara con las predicciones del Avance 5, comprobamos que la muestra y la escala de cada propiedad coinciden con las que usó aquel avance. Separamos las galaxias formadoras de las apagadas con log sSFR = −10.8, la misma frontera de los avances anteriores.

In [ ]:
CONTEXTO = preparar_contexto(BASE_DRIVE, CARPETA_RESULTADOS, OBJETIVOS, limite_galaxias=LIMITE_GALAXIAS)
CARPETA_MODELO = CONTEXTO.guardia.mkdir(BASE_DRIVE / CARPETA_RESULTADOS / NOMBRE_MODELO)
CARPETA_SALIDA = CONTEXTO.guardia.mkdir(BASE_DRIVE / CARPETA_RESULTADOS / NOMBRE_SALIDA)
CARPETA_PREDICCIONES = CONTEXTO.guardia.mkdir(CARPETA_SALIDA / "predicciones")
MUESTRA, INDICES = CONTEXTO.muestra, CONTEXTO.indices
ESCALA = CONTEXTO.desviacion_etiquetas_entrenamiento
ESCALA_AVANCE5 = predicciones.leer_escala(CARPETA_PREDICCIONES_AVANCE5)
if not np.allclose(ESCALA, ESCALA_AVANCE5, rtol=1e-6):
    raise ValueError(f"La escala de las etiquetas ({ESCALA}) no coincide con la del Avance 5 ({ESCALA_AVANCE5})")
tipo = pd.Series(np.where(MUESTRA.log_ssfr < CORTE_SSFR, "apagada", "formadora"), name="tipo de galaxia", index=MUESTRA.index)
display(pd.crosstab(MUESTRA.particion, tipo, margins=True, margins_name="total").loc[["Train", "Validation", "Test", "total"]])
print("Desviación de cada propiedad en entrenamiento:", dict(zip(OBJETIVOS, ESCALA.round(4))))

## 5. Configuración de los miembros del Avance 5

Leemos la arquitectura, los hiperparámetros y la entrada de imagen de las corridas de cada familia del Avance 5, para que las versiones encadenadas sean idénticas salvo por la cabeza. También cargamos las predicciones que esas redes exportaron para validación y prueba, que son la referencia de todas las comparaciones; no necesitamos volver a entrenar las cabezas independientes.

Para asegurarnos de copiar la configuración de las redes que realmente produjeron esas predicciones, recuperamos cada corrida guardada, volvemos a predecir la validación y la comparamos con lo exportado. Si una familia y semilla tiene más de una corrida guardada, nos quedamos con la que reproduce las predicciones. La segunda tabla muestra la diferencia máxima, que solo debe reflejar el redondeo de la GPU.

In [ ]:
TABLA_AVANCE5 = predicciones.leer_carpeta(CARPETA_PREDICCIONES_AVANCE5)
TABLA_AVANCE5 = TABLA_AVANCE5[TABLA_AVANCE5.familia.isin(FAMILIAS_BASE) & TABLA_AVANCE5.semilla.isin(SEMILLAS)]
IDS_VALIDACION = MUESTRA.mangaid.iloc[INDICES["Validation"]].to_numpy()
COLUMNAS_PRED = [f"pred_{t}" for t in OBJETIVOS]

def exportadas_avance5(familia, semilla):
    # Predicciones de validación que exportó el Avance 5, en el orden de la muestra.
    t = TABLA_AVANCE5[(TABLA_AVANCE5.familia == familia) & (TABLA_AVANCE5.semilla == semilla) & (TABLA_AVANCE5.particion == "Validation")]
    return t.set_index("mangaid").loc[IDS_VALIDACION, COLUMNAS_PRED].to_numpy(float)

def configuracion_guardada(carpeta_corrida):
    # Arquitectura, entrenamiento, imagen y variables escalares de una corrida, tal como se usaron.
    r = json.loads((Path(carpeta_corrida) / "resumen.json").read_text())
    espec = EspecificacionModelo(**{**r["especificacion"], "capas_lora": tuple(r["especificacion"].get("capas_lora", ("qkv", "proj")))})
    return espec, ConfiguracionEntrenamiento(**r["configuracion"]), r["entrada"], tuple(r.get("variables_escalares", ()))

CONFIGURACIONES, CARPETAS_AVANCE5, verificacion = {}, {}, []
for familia in FAMILIAS_BASE:
    for semilla in SEMILLAS:
        candidatas = [d for d in sorted((CARPETA_MODELO / "corridas").glob(f"{familia}_semilla{semilla}_*")) if (d / "resumen.json").exists()
                      and (d / "mejor_modelo.pt").exists()]
        if not candidatas:
            raise FileNotFoundError(f"No hay corridas terminadas de {familia} con semilla {semilla} en {CARPETA_MODELO / 'corridas'}")
        referencia = exportadas_avance5(familia, semilla)
        diferencias = {}
        for carpeta in candidatas:
            corrida = experimento.recuperar_corrida(CONTEXTO, carpeta, DISPOSITIVO)
            prediccion = experimento.predicciones_originales(CONTEXTO, corrida, "Validation", DISPOSITIVO, tta=USAR_TTA)
            diferencias[carpeta] = float(np.abs(prediccion - referencia).max())
            del corrida; liberar_gpu()
        carpeta = min(diferencias, key=diferencias.get)
        espec, config, entrada, variables = configuracion_guardada(carpeta)
        if variables != VARIABLES_ESCALARES or espec.encadenar_salidas or config.semilla != semilla:
            raise ValueError(f"{carpeta.name}: se esperaba una corrida con redshift, cabezas independientes y semilla {semilla}")
        CONFIGURACIONES[(familia, semilla)], CARPETAS_AVANCE5[(familia, semilla)] = (espec, config, entrada), carpeta
        verificacion.append({"familia": familia, "semilla": semilla, "corridas guardadas": len(candidatas), "corrida usada": carpeta.name,
                             "diferencia máxima con lo exportado": diferencias[carpeta]})
VERIFICACION_AVANCE5 = pd.DataFrame(verificacion)

filas = []
for familia in FAMILIAS_BASE:
    espec, config, entrada = CONFIGURACIONES[(familia, SEMILLAS[0])]
    filas.append({"familia": familia, "extractor": espec.nombre_extractor, "ajuste del extractor": espec.estrategia_ajuste,
                  "imagen": f"{entrada['representacion']}, {entrada['lado']} px", "épocas": config.epocas,
                  "tasa del extractor": config.tasa_extractor, "tasa de las cabezas": config.tasa_cabeza,
                  "decaimiento por capa": config.decaimiento_por_capa, "pérdida": config.perdida, "unidades por cabeza": espec.unidades_ocultas})
display(pd.DataFrame(filas).set_index("familia"))
display(VERIFICACION_AVANCE5.set_index(["familia", "semilla"]))
if VERIFICACION_AVANCE5["diferencia máxima con lo exportado"].max() > 0.02:
    raise ValueError("Alguna corrida recuperada no reproduce las predicciones exportadas en el Avance 5; revisa la tabla anterior")

## 6. Entrenamiento de las cabezas encadenadas

Entrenamos las tres familias con las cabezas en cadena, con las mismas tres semillas y las mismas reglas: se conserva la época con el menor criterio de validación y las predicciones promedian las ocho simetrías de la imagen. La cadena agrega una entrada a la cabeza de la SFR y dos a la del D4000, por lo que el número de parámetros y el tiempo por galaxia prácticamente no cambian. La figura compara las curvas de validación de cada familia con las de su versión independiente.

In [ ]:
PRED, HISTORIALES, TIEMPOS = {}, {}, []
escribir_csv = lambda ruta, f: CONTEXTO.guardia.write(ruta, f, allow_replace=True)
for familia, nueva in FAMILIAS_NUEVAS.items():
    for semilla in SEMILLAS:
        espec, config, entrada = CONFIGURACIONES[(familia, semilla)]
        espec = replace(espec, encadenar_salidas=ENCADENAR, cadena_con_gradiente=CON_GRADIENTE)
        # Con ENCADENAR = False el prefijo y la configuración son los del Avance 5: la corrida se recupera sin entrenar.
        corrida = experimento.ejecutar_corrida(CONTEXTO, CARPETA_MODELO, nueva if ENCADENAR else familia, espec, config,
                                               entrada, DISPOSITIVO, VARIABLES_ESCALARES, registro=registro_compacto)
        if not ENCADENAR and corrida.carpeta != CARPETAS_AVANCE5[(familia, semilla)]:
            raise ValueError(f"La verificación debía recuperar {CARPETAS_AVANCE5[(familia, semilla)].name} y obtuvo {corrida.carpeta.name}")
        PRED[(nueva, semilla)] = {p: experimento.predicciones_originales(CONTEXTO, corrida, p, DISPOSITIVO, tta=USAR_TTA)
                                  for p in ("Validation", "Test")}
        tabla = predicciones.tabla_predicciones(MUESTRA, INDICES, nueva, semilla, PRED[(nueva, semilla)], USAR_TTA)
        predicciones.guardar_tabla(tabla, CARPETA_PREDICCIONES, nueva, semilla, escribir_csv)
        HISTORIALES[(nueva, semilla)] = corrida.historial
        TIEMPOS.append({"familia": nueva, "semilla": semilla, "entrenada en esta sesión": not corrida.reutilizada,
                        "mejor época": corrida.resumen["mejor_epoca"], "min por entrenamiento": corrida.resumen["minutos_entrenamiento"],
                        "parámetros (M)": corrida.resumen["parametros_totales"] / 1e6,
                        "ms por galaxia": experimento.tiempo_inferencia_ms(CONTEXTO, corrida, DISPOSITIVO) if semilla == SEMILLAS[0] else np.nan})
        print(f"{nueva} semilla {semilla}: mejor época {corrida.resumen['mejor_epoca']} · criterio de validación sin TTA "
              f"{corrida.resumen['mejor_criterio_validacion']:.4f}")
        del corrida; liberar_gpu()
TIEMPOS_TABLA = pd.DataFrame(TIEMPOS)
display(TIEMPOS_TABLA.groupby("familia").agg({"mejor época": "mean", "min por entrenamiento": "mean", "parámetros (M)": "first",
                                              "ms por galaxia": "first"}).round(3))

In [ ]:
fig, ejes = plt.subplots(1, len(FAMILIAS_NUEVAS), figsize=(14, 3.6))
for eje, (familia, nueva) in zip(np.atleast_1d(ejes), FAMILIAS_NUEVAS.items()):
    independientes = [pd.read_csv(CARPETAS_AVANCE5[(familia, s)] / "historial.csv").criterio_validacion.to_numpy() for s in SEMILLAS]
    nuevas = [HISTORIALES[(nueva, s)].criterio_validacion.to_numpy() for s in SEMILLAS]
    for curvas, color, etiqueta in ((independientes, "0.45", "cabezas independientes"), (nuevas, "#2a9d8f", f"cabezas {ETIQUETA_NUEVA}")):
        n = min(len(v) for v in curvas)
        matriz = np.stack([v[:n] for v in curvas]); epocas = np.arange(1, n + 1)
        eje.plot(epocas, matriz.mean(0), color=color, label=etiqueta)
        eje.fill_between(epocas, matriz.min(0), matriz.max(0), color=color, alpha=0.2)
    eje.set(xlabel="época", ylabel="criterio de validación (sin TTA)", yscale="log", title=familia); eje.legend(fontsize=7)
plt.suptitle("Curvas de validación (línea: media de las semillas; franja: mínimo y máximo)", y=1.03); plt.tight_layout(); plt.show()

_Se completa con los resultados de la corrida._

## 7. Cada familia frente a su versión con cabezas independientes

Alineamos por galaxia las predicciones de las seis familias y comparamos cada familia encadenada con su versión independiente. Usamos el promedio de las tres semillas de cada familia, que es lo que entra al ensamble, y reportamos también el criterio medio de las semillas por separado. La diferencia de RMSE de cada propiedad lleva su intervalo de confianza del 95 % por *bootstrap* pareado y se clasifica con la regla del proyecto: **mejor** o **peor** si el intervalo excluye el cero, **equivalente** si cabe dentro del margen despreciable y **no concluyente** en otro caso.

In [ ]:
TABLA_NUEVA = predicciones.leer_carpeta(CARPETA_PREDICCIONES)
TABLA_NUEVA = TABLA_NUEVA[TABLA_NUEVA.familia.isin(FAMILIAS_NUEVAS.values()) & TABLA_NUEVA.semilla.isin(SEMILLAS)]
TABLA = pd.concat([TABLA_AVANCE5, TABLA_NUEVA], ignore_index=True)
TODAS = list(FAMILIAS_BASE) + list(FAMILIAS_NUEVAS.values())
VAL = predicciones.construir_conjunto(TABLA, "Validation", TODAS)
PRUEBA = predicciones.construir_conjunto(TABLA, "Test", TODAS)
Y_VAL, Y_PRUEBA = VAL.y, PRUEBA.y
print(f"Galaxias alineadas: {len(VAL.ids)} de validación y {len(PRUEBA.ids)} de prueba")

def rmse(y, p):
    return np.sqrt(np.mean((np.asarray(p) - np.asarray(y)) ** 2, axis=0))

filas, diferencias = [], []
for familia, nueva in FAMILIAS_NUEVAS.items():
    for cabezas, f in (("independientes", familia), (ETIQUETA_NUEVA, nueva)):
        filas.append({"familia": familia, "cabezas": cabezas, "criterio (promedio de semillas)": ensambles.criterio(Y_VAL, VAL.media(f), ESCALA),
                      "criterio medio por semilla": np.mean([ensambles.criterio(Y_VAL, q, ESCALA) for q in VAL.pred[f]]),
                      **{f"RMSE {CORTO[t]}": v for t, v in zip(OBJETIVOS, rmse(Y_VAL, VAL.media(f)))}})
    d, lo, hi = ensambles.diferencia_pareada_criterio(Y_VAL, VAL.media(nueva), VAL.media(familia), ESCALA, N_REMUESTREOS)
    diferencias.append({"familia": familia, "medida": "criterio", "diferencia (cadena − independientes)": d, "IC 95 % inferior": lo,
                        "IC 95 % superior": hi, "veredicto": "mejor" if hi < 0 else ("peor" if lo > 0 else "no se distinguen")})
    for k, t in enumerate(OBJETIVOS):
        d, lo, hi = evaluacion.diferencia_pareada_rmse(Y_VAL[:, k], VAL.media(nueva)[:, k], VAL.media(familia)[:, k], n_remuestreos=N_REMUESTREOS)
        diferencias.append({"familia": familia, "medida": f"RMSE {CORTO[t]}", "diferencia (cadena − independientes)": d, "IC 95 % inferior": lo,
                            "IC 95 % superior": hi, "veredicto": evaluacion.veredicto(lo, hi, MARGEN[CORTO[t]])})
COMPARACION_FAMILIAS = pd.DataFrame(filas)
DIFERENCIAS_FAMILIAS = pd.DataFrame(diferencias)
display(COMPARACION_FAMILIAS.set_index(["familia", "cabezas"]).round(4))
display(DIFERENCIAS_FAMILIAS.set_index(["familia", "medida"]).round(4))

In [ ]:
medidas = [f"RMSE {CORTO[t]}" for t in OBJETIVOS]
fig, ejes = plt.subplots(1, 3, figsize=(14, 3.2), sharey=True)
for eje, medida, t in zip(ejes, medidas, OBJETIVOS):
    sub = DIFERENCIAS_FAMILIAS[DIFERENCIAS_FAMILIAS.medida == medida].reset_index(drop=True)
    y = np.arange(len(sub))[::-1]
    d = sub["diferencia (cadena − independientes)"]
    eje.errorbar(d, y, xerr=[d - sub["IC 95 % inferior"], sub["IC 95 % superior"] - d], fmt="o", color="#264653", capsize=3)
    eje.axvline(0, color="k", lw=0.8)
    eje.axvspan(-MARGEN[CORTO[t]], MARGEN[CORTO[t]], color="0.85", zorder=0, label="margen despreciable")
    eje.set(yticks=y, yticklabels=sub.familia, xlabel=f"ΔRMSE {NOMBRES[t]}", title=CORTO[t])
ejes[0].legend(fontsize=7, loc="lower left")
plt.suptitle("Cambio del RMSE de validación al encadenar las cabezas (negativo: la cadena reduce el error; barras: IC 95 %)", y=1.04)
plt.tight_layout(); plt.show()

_Se completa con los resultados de la corrida._

## 8. ¿Dónde actúa la cadena?

Si la cadena ayuda por la razón física que la motiva, el efecto debería concentrarse en la SFR, y en particular en las galaxias apagadas, donde el modelo del Avance 5 comprime sus estimaciones hacia valores centrales (pendiente de 0.79 en prueba). Por eso separamos el RMSE de la SFR por tipo de galaxia y medimos la pendiente de la predicción frente a la etiqueta en la SFR y el D4000.

La cadena también tiene un costo posible: si la cabeza de la SFR usa la masa estimada, un error en la masa se propaga a la SFR. Lo medimos con la correlación entre el residuo de la masa y el de la SFR; si la cadena la aumenta, los errores de las dos propiedades dejan de ser independientes, lo que importa para quien use el catálogo en relaciones como la secuencia principal.

In [ ]:
APAGADA_VAL = (Y_VAL[:, 1] - Y_VAL[:, 0]) < CORTE_SSFR
filas = []
for familia, nueva in FAMILIAS_NUEVAS.items():
    for cabezas, f in (("independientes", familia), (ETIQUETA_NUEVA, nueva)):
        p = VAL.media(f); r = p - Y_VAL
        filas.append({"familia": familia, "cabezas": cabezas,
                      "RMSE SFR formadoras": rmse(Y_VAL[~APAGADA_VAL, 1], p[~APAGADA_VAL, 1]),
                      "RMSE SFR apagadas": rmse(Y_VAL[APAGADA_VAL, 1], p[APAGADA_VAL, 1]),
                      "pendiente SFR": ensambles.pendiente_de_calibracion(Y_VAL[:, 1], p[:, 1]),
                      "pendiente D4000": ensambles.pendiente_de_calibracion(Y_VAL[:, 2], p[:, 2]),
                      "correlación de residuos M*–SFR": np.corrcoef(r[:, 0], r[:, 1])[0, 1],
                      "correlación de residuos M*–D4000": np.corrcoef(r[:, 0], r[:, 2])[0, 1]})
DIAGNOSTICO = pd.DataFrame(filas)
display(DIAGNOSTICO.set_index(["familia", "cabezas"]).round(3))
print(f"Galaxias de validación: {int((~APAGADA_VAL).sum())} formadoras y {int(APAGADA_VAL.sum())} apagadas")

_Se completa con los resultados de la corrida._

## 9. Ensambles

Reconstruimos la mezcla convexa con exactamente el procedimiento del Avance 5: pesos no negativos que suman uno en cada propiedad, una regularización que los acerca al promedio uniforme y la elección de esa regularización por validación anidada (cinco pliegues externos repetidos tres veces y cuatro internos). La validación anidada mide el desempeño de cada mezcla con galaxias que no intervinieron en sus pesos. Construimos tres mezclas: la del Avance 5 con cabezas independientes, que debe reproducir su criterio de 0.1037; la de las tres familias encadenadas; y la de las seis familias juntas, por si las versiones independientes y encadenadas se complementan.

In [ ]:
ENSAMBLES = {"Mezcla convexa, cabezas independientes (Avance 5)": list(FAMILIAS_BASE),
             f"Mezcla convexa, cabezas {ETIQUETA_NUEVA}": list(FAMILIAS_NUEVAS.values()),
             "Mezcla convexa de las seis familias": TODAS}
INCUMBENTE, CANDIDATO_CADENA = list(ENSAMBLES)[:2]
RESULTADOS_ENSAMBLES = {}
for nombre, familias in ENSAMBLES.items():
    P = VAL.medias(familias)
    anidada = ensambles.validacion_anidada(P, Y_VAL, None, ESCALA, candidatos=CANDIDATOS_MEZCLA, pliegues_externos=PLIEGUES_EXTERNOS,
                                           repeticiones=REPETICIONES, pliegues_internos=PLIEGUES_INTERNOS, semilla=SEMILLA_ANIDADA,
                                           n_trabajos=N_TRABAJOS)
    config, _ = ensambles.buscar_configuracion(P, Y_VAL, None, ESCALA, candidatos=CANDIDATOS_MEZCLA, pliegues_internos=PLIEGUES_EXTERNOS,
                                               semilla=SEMILLA_ANIDADA)
    pesos = ensambles.pesos_mezcla_convexa(P, Y_VAL, ESCALA, config["lambda"], config["alcance"])
    RESULTADOS_ENSAMBLES[nombre] = {"familias": familias, "anidada": anidada, "configuracion": config, "pesos": pesos, "redes": len(SEMILLAS) * len(familias)}

filas = []
for nombre, r in RESULTADOS_ENSAMBLES.items():
    oof = r["anidada"]["oof_por_repeticion"]
    filas.append({"ensamble": nombre, "redes por galaxia": r["redes"], "criterio (validación anidada)": r["anidada"]["criterio_medio"],
                  "desviación entre repeticiones": r["anidada"]["criterio_desviacion"],
                  **{f"RMSE {CORTO[t]}": np.mean([rmse(Y_VAL, o)[k] for o in oof]) for k, t in enumerate(OBJETIVOS)},
                  "regularización elegida": f"{r['configuracion']['alcance']}, λ = {r['configuracion']['lambda']}"})
TABLA_ENSAMBLES = pd.DataFrame(filas).set_index("ensamble")
display(TABLA_ENSAMBLES.round(4))

if RESUMEN_HETEROGENEOS_AVANCE5.exists():
    referencia = json.loads(RESUMEN_HETEROGENEOS_AVANCE5.read_text())["h1_mezcla_convexa"]
    diferencia_criterio = RESULTADOS_ENSAMBLES[INCUMBENTE]["anidada"]["criterio_medio"] - referencia["criterio_medio"]
    diferencia_pesos = float(np.abs(RESULTADOS_ENSAMBLES[INCUMBENTE]["pesos"] - np.asarray(referencia["pesos"])).max())
    print(f"Comprobación con el Avance 5: criterio {RESULTADOS_ENSAMBLES[INCUMBENTE]['anidada']['criterio_medio']:.4f} "
          f"(reportado {referencia['criterio_medio']:.4f}) · diferencia máxima de los pesos {diferencia_pesos:.4f}")
    if abs(diferencia_criterio) > 1e-3 or diferencia_pesos > 0.02:
        print("AVISO: la mezcla del Avance 5 no se reproduce; revisa que las predicciones y la muestra sean las mismas.")

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 0.9 + 0.32 * len(TODAS)), sharey=True)
for eje, (nombre, r) in zip(ejes, RESULTADOS_ENSAMBLES.items()):
    pesos = pd.DataFrame(r["pesos"], index=r["familias"], columns=[CORTO[t] for t in OBJETIVOS]).reindex(TODAS)
    imagen = eje.imshow(pesos.fillna(0).to_numpy(), vmin=0, vmax=1, cmap="Greens", aspect="auto")
    for i, f in enumerate(TODAS):
        for j in range(len(OBJETIVOS)):
            valor = pesos.iloc[i, j]
            eje.text(j, i, "·" if np.isnan(valor) else f"{valor:.2f}", ha="center", va="center", fontsize=8, color="k" if np.isnan(valor) or valor < 0.6 else "w")
    eje.set(xticks=range(len(OBJETIVOS)), xticklabels=pesos.columns, yticks=range(len(TODAS)), yticklabels=TODAS, title=nombre)
    eje.title.set_fontsize(8)
fig.colorbar(imagen, ax=ejes, shrink=0.8, pad=0.02, label="peso en la mezcla")
plt.suptitle("Pesos de cada familia por propiedad (ajustados con toda la validación)", y=1.03); plt.show()

_Se completa con los resultados de la corrida._

## 10. Selección y registro antes de abrir la prueba

Cambiar el modelo final exige una mejora que no se explique por el azar de la muestra de validación. Comparamos cada candidato con la mezcla del Avance 5 usando la pérdida de cada galaxia en la validación anidada, promediada entre las repeticiones; como todos los modelos se evalúan sobre las mismas galaxias, el error estándar que importa es el de la diferencia pareada. Un candidato reemplaza al modelo del Avance 5 solo si lo supera por más de ese error estándar. Si varios lo logran, aplicamos la regla de un error estándar del proyecto (Hastie et al. 2009) y elegimos el que necesite menos redes por galaxia. La elección queda registrada, con una huella de sus predicciones de validación, antes de cualquier evaluación en prueba.

In [ ]:
# Pérdida de cada galaxia, promediada entre las repeticiones de la validación anidada (mismas particiones en todos los ensambles).
PERDIDAS = {n: np.mean([ensambles.perdida_por_galaxia(Y_VAL, o, ESCALA) for o in r["anidada"]["oof_por_repeticion"]], axis=0)
            for n, r in RESULTADOS_ENSAMBLES.items()}
filas = []
for nombre, perdida in PERDIDAS.items():
    d = perdida - PERDIDAS[INCUMBENTE]
    error = float(np.std(d, ddof=1) / np.sqrt(len(d))) if nombre != INCUMBENTE else np.nan
    filas.append({"ensamble": nombre, "criterio (media de las repeticiones)": float(perdida.mean()), "diferencia con el Avance 5": float(d.mean()),
                  "error estándar de la diferencia": error, "redes por galaxia": RESULTADOS_ENSAMBLES[nombre]["redes"],
                  # el umbral mínimo evita declarar ganador a un ensamble idéntico por redondeo (p. ej., con ENCADENAR = False)
                  "supera al Avance 5": bool(nombre != INCUMBENTE and d.mean() < -max(error, 1e-6))})
SELECCION = pd.DataFrame(filas).set_index("ensamble")
display(SELECCION.round(5))

superan = list(SELECCION.index[SELECCION["supera al Avance 5"]])
if superan:
    ELEGIDO, TABLA_REGLA, _ = ensambles.regla_un_error_estandar({n: PERDIDAS[n] for n in superan},
                                                                {n: RESULTADOS_ENSAMBLES[n]["redes"] for n in superan})
else:
    ELEGIDO = INCUMBENTE
print("Modelo elegido:", ELEGIDO)

P_VAL_ELEGIDO = ensambles.aplicar_pesos(VAL.medias(RESULTADOS_ENSAMBLES[ELEGIDO]["familias"]), RESULTADOS_ENSAMBLES[ELEGIDO]["pesos"])
REGISTRO = ensambles.registrar_eleccion(CARPETA_SALIDA / f"registro_eleccion{SUFIJO}.json",
                                        {"modelo": ELEGIDO, "familias": RESULTADOS_ENSAMBLES[ELEGIDO]["familias"],
                                         "configuracion": RESULTADOS_ENSAMBLES[ELEGIDO]["configuracion"],
                                         "pesos": RESULTADOS_ENSAMBLES[ELEGIDO]["pesos"].tolist(),
                                         "criterio_validacion": RESULTADOS_ENSAMBLES[ELEGIDO]["anidada"]["criterio_medio"],
                                         "encadenar": ENCADENAR, "con_gradiente": CON_GRADIENTE, "commit": COMMIT}, P_VAL_ELEGIDO)

_Se completa con los resultados de la corrida._

## 11. Evaluación en prueba

Si el modelo elegido sigue siendo el del Avance 5, no reabrimos la prueba: su resultado ya está reportado (criterio de 0.1102). Si un candidato lo superó en validación, lo evaluamos una sola vez en las 917 galaxias de prueba, junto con el modelo del Avance 5 con sus pesos originales, y medimos la diferencia con *bootstrap* pareado. Como la prueba ya se usó una vez, este resultado confirma o matiza el de validación, pero no sustituye a una prueba independiente.

In [ ]:
if ELEGIDO == INCUMBENTE:
    print("Se conserva el modelo del Avance 5; no se reabre la prueba.")
elif not EVALUAR_PRUEBA:
    print("EVALUAR_PRUEBA = False: el modelo elegido queda registrado sin evaluar en prueba.")
else:
    pesos_avance5 = (np.asarray(json.loads(RESUMEN_HETEROGENEOS_AVANCE5.read_text())["h1_mezcla_convexa"]["pesos"])
                     if RESUMEN_HETEROGENEOS_AVANCE5.exists() else RESULTADOS_ENSAMBLES[INCUMBENTE]["pesos"])
    P_PRUEBA = {INCUMBENTE: ensambles.aplicar_pesos(PRUEBA.medias(list(FAMILIAS_BASE)), pesos_avance5),
                ELEGIDO: ensambles.aplicar_pesos(PRUEBA.medias(RESULTADOS_ENSAMBLES[ELEGIDO]["familias"]), RESULTADOS_ENSAMBLES[ELEGIDO]["pesos"])}
    filas = []
    for nombre, p in P_PRUEBA.items():
        metricas = {t: evaluacion.metricas(Y_PRUEBA[:, k], p[:, k]) for k, t in enumerate(OBJETIVOS)}
        filas.append({"modelo": nombre, "criterio": ensambles.criterio(Y_PRUEBA, p, ESCALA),
                      **{f"{m} {CORTO[t]}": metricas[t][m] for t in OBJETIVOS for m in ("RMSE", "MAE", "R2")}})
    display(pd.DataFrame(filas).set_index("modelo").round(4))
    d, lo, hi = ensambles.diferencia_pareada_criterio(Y_PRUEBA, P_PRUEBA[ELEGIDO], P_PRUEBA[INCUMBENTE], ESCALA, N_REMUESTREOS)
    print(f"Diferencia de criterio en prueba (elegido − Avance 5): {d:+.4f}, IC 95 % [{lo:+.4f}, {hi:+.4f}]")

_Se completa con los resultados de la corrida._

## 12. Resultados exportados

Guardamos las predicciones de validación y prueba de las familias encadenadas en el mismo formato común del Avance 5, junto con un resumen de los ensambles y de la elección, para que el experimento se pueda revisar sin GPU.

In [ ]:
RESUMEN = {"encadenar": ENCADENAR, "con_gradiente": CON_GRADIENTE, "familias": FAMILIAS_NUEVAS, "semillas": list(SEMILLAS), "tta": USAR_TTA,
           "commit": COMMIT, "gpu": NOMBRE_GPU, "fecha_utc": datetime.now(timezone.utc).isoformat(), "elegido": ELEGIDO,
           "galaxias": {"Validation": int(len(VAL.ids)), "Test": int(len(PRUEBA.ids))},
           "ensambles": {n: {"familias": r["familias"], "configuracion": r["configuracion"], "pesos": r["pesos"].tolist(),
                             "criterios": r["anidada"]["criterios"], "criterio_medio": r["anidada"]["criterio_medio"]}
                         for n, r in RESULTADOS_ENSAMBLES.items()},
           "familias_validacion": COMPARACION_FAMILIAS.to_dict("records"), "diferencias_validacion": DIFERENCIAS_FAMILIAS.to_dict("records")}
CONTEXTO.guardia.write_json(CARPETA_SALIDA / f"resumen{SUFIJO}.json", json.loads(json.dumps(RESUMEN, default=float)), allow_replace=True)
CONTEXTO.guardia.write(CARPETA_PREDICCIONES / f"tiempos{SUFIJO}.csv", lambda f: TIEMPOS_TABLA.to_csv(f, index=False), allow_replace=True)
RUTA_ZIP = CARPETA_SALIDA / f"experimento{SUFIJO}.zip"
def escribir_zip(destino):
    with zipfile.ZipFile(destino, "w", zipfile.ZIP_DEFLATED) as z:
        nombres = [predicciones.nombre_archivo(nueva, s) for nueva in FAMILIAS_NUEVAS.values() for s in SEMILLAS] + [f"tiempos{SUFIJO}.csv"]
        for nombre in nombres:
            z.write(CARPETA_PREDICCIONES / nombre, f"predicciones/{nombre}")
        for archivo in (CARPETA_SALIDA / f"resumen{SUFIJO}.json", CARPETA_SALIDA / f"registro_eleccion{SUFIJO}.json"):
            if archivo.exists():
                z.write(archivo, archivo.name)
CONTEXTO.guardia.write(RUTA_ZIP, escribir_zip, allow_replace=True)
print(f"{len(TABLA_NUEVA.groupby(['familia', 'semilla']))} redes exportadas")
if EN_COLAB:
    from google.colab import files
    files.download(str(RUTA_ZIP))

## 13. Conclusiones

_Se completa con los resultados de la corrida._

## Referencias

Alfonzo, J. P., Iyer, K. G., Akiyama, M., et al. (2024). Katachi: Decoding the imprints of past star formation on present-day morphology in galaxies with interpretable CNNs. *The Astrophysical Journal, 967*(2), 152. https://doi.org/10.3847/1538-4357/ad3b95

Hastie, T., Tibshirani, R. y Friedman, J. (2009). *The elements of statistical learning* (2.ª ed.). Springer.

Studer, S., Bui, T. B., Drescher, C., et al. (2021). Towards CRISP-ML(Q): A machine learning process model with quality assurance methodology. *Machine Learning and Knowledge Extraction, 3*(2), 392–413.